# CC2017 – Deep Learning · Proyecto Final
## 03 · Transfer learning y fine-tuning con MobileNetV2

MobileNetV2 fue preentrenado en ImageNet (1.2 millones de imágenes, 1,000 clases). Se reemplaza su clasificador por uno de 43 clases y se adapta en dos etapas:
- **Etapa A · extracción de características:** el backbone queda congelado y solo se entrena el clasificador nuevo.
- **Etapa B · fine-tuning:** se descongelan los últimos bloques y se entrenan con una tasa de aprendizaje 10 veces menor, para ajustar las características sin destruir lo aprendido.

## Preparación · Librerías, reproducibilidad y dispositivo

In [ ]:
import sys, random, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
OUT = ROOT / 'outputs'; OUT.mkdir(exist_ok=True)

# FAST_DEV_RUN=True recorre pocos lotes por época para comprobar que todo corre; sus checkpoints llevan prefijo dev_.
FAST_DEV_RUN = False
MAX_BATCHES = 5 if FAST_DEV_RUN else None
PREFIX = 'dev_' if FAST_DEV_RUN else ''
NUM_WORKERS = 2
print('Dispositivo:', device, '| FAST_DEV_RUN:', FAST_DEV_RUN)

## 1. Datos
Se usa 96×96 en lugar de 224×224 para que el entrenamiento sea viable en CPU. Las imágenes se normalizan con la media y desviación de ImageNet, que es lo que espera el backbone.

In [ ]:
from src.data import download_gtsrb, load_tables, split_train_val, GTSRBDataset, get_transforms, CLASS_NAMES, NUM_CLASSES

DATA_ROOT = download_gtsrb()
train_full, test_df = load_tables(DATA_ROOT)
train_df, val_df = split_train_val(train_full, val_size=0.15, seed=SEED)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

In [ ]:
IMG_SIZE = 96
batch_size = 64

train_ds = GTSRBDataset(train_df, DATA_ROOT, get_transforms(IMG_SIZE, augment=True, imagenet_norm=True))
val_ds = GTSRBDataset(val_df, DATA_ROOT, get_transforms(IMG_SIZE, imagenet_norm=True))
train_loader = DataLoader(train_ds, batch_size, shuffle=True, num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(val_ds, batch_size, num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0)

## 2. Etapa A · Backbone congelado

In [ ]:
from src.models import build_mobilenet_v2, unfreeze_last_blocks, count_params
from src.train import fit

mnet = build_mobilenet_v2(freeze_backbone=True).to(device)
total, trainable_a = count_params(mnet)
print(f'Parámetros: {total:,} | entrenables en etapa A: {trainable_a:,} ({trainable_a/total:.1%})')

opt_a = torch.optim.Adam([p for p in mnet.parameters() if p.requires_grad], lr=1e-3)
sched_a = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_a, factor=0.5, patience=2)
hist_a = fit(mnet, train_loader, val_loader, opt_a, device, epochs=2 if FAST_DEV_RUN else 10,
             scheduler=sched_a, patience=3, ckpt_path=OUT / f'{PREFIX}mobilenet_frozen.pt', max_batches=MAX_BATCHES)

## 3. Etapa B · Fine-tuning
Se parte de los pesos de la etapa A y se descongelan los últimos 4 bloques de `features`. Se usan dos grupos de parámetros: lr 1e-4 para el backbone y 1e-3 para el clasificador. Para el avance esta etapa es opcional: con `RUN_FINETUNE = False` se omite.

In [ ]:
RUN_FINETUNE = True

if RUN_FINETUNE:
    mnet_ft = build_mobilenet_v2(freeze_backbone=True).to(device)
    mnet_ft.load_state_dict(mnet.state_dict())
    unfreeze_last_blocks(mnet_ft, n_blocks=4)
    _, trainable_b = count_params(mnet_ft)
    print(f'Entrenables en etapa B: {trainable_b:,} ({trainable_b/total:.1%})')
    backbone = [p for p in mnet_ft.features.parameters() if p.requires_grad]
    opt_b = torch.optim.Adam([{'params': backbone, 'lr': 1e-4},
                              {'params': mnet_ft.classifier.parameters(), 'lr': 1e-3}])
    sched_b = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_b, factor=0.5, patience=2)
    hist_b = fit(mnet_ft, train_loader, val_loader, opt_b, device, epochs=2 if FAST_DEV_RUN else 10,
                 scheduler=sched_b, patience=3, ckpt_path=OUT / f'{PREFIX}mobilenet_finetuned.pt', max_batches=MAX_BATCHES)

## 4. Comparación de etapas

In [ ]:
from src.metrics import predict, summary_metrics
stages = [('MobileNetV2 congelado', mnet, hist_a, trainable_a)]
if RUN_FINETUNE: stages.append(('MobileNetV2 fine-tuning', mnet_ft, hist_b, trainable_b))
rows = []
for name, model, h, n_train in stages:
    y, p, _ = predict(model, val_loader, device, MAX_BATCHES)
    rows.append({**summary_metrics(y, p, name), 'params_entrenables': n_train,
                 'épocas': len(h['val_loss']), 's_por_época': np.mean(h['epoch_time'])})
display(pd.DataFrame(rows).round(4))

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for name, _, h, _ in stages:
    axs[0].plot(h['val_loss'], label=name); axs[1].plot(h['val_acc'], label=name)
axs[0].set(title='Pérdida de validación', xlabel='Época'); axs[1].set(title='Accuracy de validación', xlabel='Época')
for ax in axs: ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT / 'curvas_mobilenet.png', dpi=120); plt.show()

**Interpretación (completar después de ejecutar):** compare la accuracy y F1 de ambas etapas con el porcentaje de parámetros entrenables y el tiempo por época. ¿Las características de ImageNet bastan con el backbone congelado, o el fine-tuning aporta una mejora clara? Registre ambos resultados en `docs/bitacora_iteraciones.md`.